# LLM으로 지식 그래프(Knowledge Graph) 구축

**코스 모듈:** Module 8
**대상:** 그래프 기초, Neo4j 기본, 생성형 AI / LLM, 간단한 Cypher를 이미 이해한 초급 학습자

**코스 설명**

이 실습 코스에서는 **대규모 언어 모델(LLM, Large Language Model)**을 사용해 지식 그래프를 생성하고 조회하는 방법을 배웁니다. 비정형 데이터에서 지식 그래프를 구축하기 위해 **Neo4j LLM Knowledge Graph Builder**를 사용한 뒤, 데이터 모델을 맞춤 설정하고 **Cypher**로 그래프를 조회합니다.

**학습 내용**

1. 지식 그래프란 무엇인지, GenAI 애플리케이션(GraphRAG, 근거 기반 답변)에 어떻게 도움이 되는지
2. LLM이 비정형 텍스트에서 **엔티티(entity)**와 **관계(relationship)**를 추출하는 방법
3. 빠른 프로토타이핑을 위한 **LLM Knowledge Graph Builder**(웹 앱) 사용법
4. Python으로 그래프를 **프로그래밍 방식**으로 구축·적재하는 방법(웹 앱과 동일한 개념)
5. **스키마로 추출을 안내**하고 Cypher로 결과를 **조회**하는 방법

> **언어:** 이 노트북의 안내 텍스트는 **한국어**입니다. 기술 용어는 필요 시 영어를 병기합니다.

> **설정:** 코드 셀을 실행하기 전에 이 폴더의 **`NEO4J_SETUP.md`**(Neo4j/데이터베이스)와 **`LLM_MODEL_SETUP.md`**(`ollama_model_runner.py` 또는 OpenAI를 통한 LLM)를 완료하세요.


## 코스 개요

| 파트 | 주제 |
|------|--------|
| **01** | 지식 그래프 |
| **02** | LLM Graph Builder |
| **03** | 지식 그래프 탐색 |
| 0 | 환경 및 Neo4j 연결 |

### 01. 지식 그래프

| 섹션 | 주제 |
|---------|--------|
| 1.1 | 지식 그래프란 |
| 1.2 | 이점과 과제 |
| 1.3 | 지식 그래프 탐색 |
| 1.4 | 지식 그래프 활용 사례 |

### 02. LLM Graph Builder

| 섹션 | 주제 |
|---------|--------|
| 2.1 | LLM으로 지식 그래프 구축하는 방법 |
| 2.2 | Neo4j LLM Graph Builder |
| 2.3 | 스키마 맞춤 설정 |
| 2.4 | 그래프 완성 |

### 03. 지식 그래프 탐색

| 섹션 | 주제 |
|---------|--------|
| 3.1 | Cypher로 조회 |
| 3.2 | 지식 그래프 탐색 |
| 3.3 | 나만의 지식 그래프 만들기


## Step 0 — 환경 및 Neo4j 연결

### 코드 실행 전

1. **`NEO4J_SETUP.md`** 완료 — Neo4j **5.15+** 데이터베이스 설치 및 시작(Aura, Desktop, Docker 중 선택)
2. **`LLM_MODEL_SETUP.md`** 완료 — **Ollama** + `ollama_model_runner.py`(권장) 또는 **OpenAI** 구성
3. `Module_8/.env.example` → `Module_8/.env` 복사 후 자격 증명 입력

아래 코드 셀을 실행하려면 **실행 중인** 데이터베이스와 연결 URI, 사용자명, 비밀번호가 필요합니다.

### Neo4j 데이터베이스 시작 및 Browser 열기

이 노트북은 **Bolt** 프로토콜(포트 **7687**)로 연결합니다. 파트 1–3에서 그래프를 시각화하려면 **Neo4j Browser**(로컬 환경에서는 포트 **7474**)도 사용합니다.

배포 방식에 맞는 행을 선택하세요(전체 설치 절차는 **`NEO4J_SETUP.md`** 참고):

| 배포 방식 | 데이터베이스 시작 | Neo4j Browser 열기 |
|------------|-------------------|-------------------|
| **Aura (클라우드)** | [Neo4j Aura 콘솔](https://console.neo4j.io/)에서 인스턴스 상태가 **Running**인지 확인 | 인스턴스에서 **Open** / **Query** 클릭 |
| **Neo4j Desktop** | 인스턴스 선택 → **Start** 클릭(상태 **Running**) | 인스턴스에서 **Open** 클릭 |
| **Docker** | `NEO4J_SETUP.md` **Option C** 참고 — `docker ps`에 컨테이너가 보여야 함 | [http://localhost:7474](http://localhost:7474) |

**Neo4j Browser 로그인(Desktop / Docker 첫 로그인):**

1. 위 표의 Browser URL을 엽니다.
2. **Connect URL:** `neo4j://localhost:7687`(로컬) 또는 Aura URI(`neo4j+s://…`)
3. **Username:** `neo4j`
4. **Password:** 인스턴스 생성 시 설정한 비밀번호 — `Module_8/.env`의 `NEO4J_PASSWORD`와 일치해야 합니다.

**스모크 테스트** — 노트북을 계속하기 전에 Neo4j Browser에서 실행:

```cypher
RETURN "Neo4j connection OK" AS message;
```

해당 메시지가 한 행으로 표시되어야 합니다. 로그인이 실패하면 `.env`의 비밀번호를 인스턴스 설정과 대조하세요(Docker는 `NEO4J_AUTH`와 일치해야 함).

**`.env` 변수**(아래 셀에서 사용):

| 변수 | 로컬(Desktop / Docker) | Aura 클라우드 |
|----------|--------------------------|------------|
| `NEO4J_URI` | `neo4j://localhost:7687` | `neo4j+s://xxxx.databases.neo4j.io` |
| `NEO4J_USERNAME` | `neo4j` | `neo4j` |
| `NEO4J_PASSWORD` | 인스턴스 비밀번호 | Aura 자격 증명 파일에서 |
| `NEO4J_DATABASE` | `neo4j` | `neo4j` |

> **보안:** 실제 비밀번호를 Git에 커밋하지 마세요. `.env`만 사용하세요(gitignore됨).

### 이 단계에서 하는 일

1. Python 패키지 설치
2. `Module_8/.env`에서 Neo4j 및 LLM 설정 로드
3. Neo4j 연결 확인(Bolt)
4. `LLMGraphTransformer`용 LLM 구성(Step **0d.1–0d.4**):
   - **`LLM_BACKEND=ollama`** → 서브프로세스로 `ollama_model_runner.py` 호출
   - **`LLM_BACKEND=openai`** → `ChatOpenAI`


In [1]:
# Step 0a — Install Python dependencies (run once per environment)
%pip install -q neo4j python-dotenv requests \
    langchain langchain-community langchain-experimental langchain-neo4j \
    langchain-openai sentence-transformers


Note: you may need to restart the kernel to use updated packages.


### Step 0b.1 — `Module_8` 폴더 경로 확인
Jupyter는 저장소 루트 또는 `Module_8` 내부에서 시작될 수 있습니다. 이 셀은 `.env`와 `data/`가 있는 폴더를 찾습니다.


In [2]:
# Step 0b.1 — Resolve Module_8 directory
import os
from pathlib import Path
from dotenv import load_dotenv
MODULE_DIR = Path(".").resolve()
if MODULE_DIR.name != "Module_8":
    _candidate = MODULE_DIR / "Module_8"
    if _candidate.is_dir():
        MODULE_DIR = _candidate.resolve()
load_dotenv(MODULE_DIR / ".env")
print("모듈 디렉터리:", MODULE_DIR)


모듈 디렉터리: /home/ethan/newgen/KMOU_Course/Module_8


### Step 0b.2 — Neo4j 연결 설정
값은 `Module_8/.env`에서 읽습니다(**`NEO4J_SETUP.md`** 참고).


In [3]:
# Step 0b.2 — Neo4j settings
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
print("Neo4j URI:", NEO4J_URI)
print("Neo4j 데이터베이스:", NEO4J_DATABASE)


Neo4j URI: neo4j://localhost:7687
Neo4j 데이터베이스: neo4j


### Step 0b.3 — LLM 백엔드 및 랩 데이터셋
- **`LLM_BACKEND=ollama`**(기본값)은 이후 셀에서 `ollama_model_runner.py`를 사용합니다.
- **`LLM_BACKEND=openai`**는 `ChatOpenAI`와 `OPENAI_API_KEY`가 필요합니다.
기본 코퍼스: **`data/dbpedia_course_corpus.txt`** (`data/DATASETS.md` 참고).

- **`LAB_MAX_ARTICLES`**(기본값 `5`): Section 2.2b에서 LLM에 보낼 코퍼스 기사 수(100회 프롬프트/거대 단일 프롬프트 방지).
- **`LAB_SCHEMA_ARTICLES`**(기본값 `8`): Section 2.3에서 사용하는 기사 수; Port/Organization 스키마에는 해양(maritime) 항목이 가장 적합합니다.


In [4]:
# Step 0b.3 — LLM + dataset path
LLM_BACKEND = os.getenv("LLM_BACKEND", "ollama").lower()
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
OLLAMA_TEMPERATURE = float(os.getenv("OLLAMA_TEMPERATURE", "0"))
OLLAMA_MAX_TOKENS = int(os.getenv("OLLAMA_MAX_TOKENS", "2048"))
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
LAB_MAX_ARTICLES = int(os.getenv("LAB_MAX_ARTICLES", "5"))
LAB_SCHEMA_ARTICLES = int(os.getenv("LAB_SCHEMA_ARTICLES", "8"))
SAMPLE_TEXT_PATH = MODULE_DIR / "data" / "dbpedia_course_corpus.txt"
# SAMPLE_TEXT_PATH = MODULE_DIR / "data" / "dbpedia_maritime_corpus.txt"
print("LLM 백엔드:", LLM_BACKEND)
print("LAB_MAX_ARTICLES (오픈 스키마):", LAB_MAX_ARTICLES)
print("LAB_SCHEMA_ARTICLES (스키마 랩):", LAB_SCHEMA_ARTICLES)
print("데이터셋:", SAMPLE_TEXT_PATH.name, "| 존재:", SAMPLE_TEXT_PATH.is_file())
if LLM_BACKEND == "ollama":
    print("Ollama 호스트:", OLLAMA_HOST)
    print("Ollama 모델:", OLLAMA_MODEL)


LLM 백엔드: ollama
LAB_MAX_ARTICLES (오픈 스키마): 5
LAB_SCHEMA_ARTICLES (스키마 랩): 8
데이터셋: dbpedia_course_corpus.txt | 존재: True
Ollama 호스트: http://localhost:11434
Ollama 모델: llama3.1:8b


### Step 0c — Neo4j 연결 확인

공식 `neo4j` 드라이버로 짧은 **Bolt** 세션을 엽니다 — `Neo4jGraph`와 `LLMGraphTransformer`가 그래프를 적재할 때 사용하는 것과 동일한 프로토콜입니다.

**이 셀이 실패할 때:**

| 오류 | 확인 사항 |
|-------|----------------|
| `NEO4J_PASSWORD is empty` | `Module_8/.env`에 `NEO4J_PASSWORD` 입력 |
| `ServiceUnavailable` | 데이터베이스 미실행 또는 잘못된 URI(위 **Neo4j 데이터베이스 시작 및 Browser 열기** 참고) |
| `Authentication failed` | 비밀번호 불일치 — Docker `NEO4J_AUTH`와 `.env`가 일치해야 함 |


In [5]:
# Step 0c — Verify Neo4j connectivity
from neo4j import GraphDatabase

if not NEO4J_PASSWORD:
    raise ValueError(
        "NEO4J_PASSWORD is empty. Set it in your environment or Module_8/.env "
        "(see NEO4J_SETUP.md and LLM_MODEL_SETUP.md)."
    )

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with driver.session(database=NEO4J_DATABASE) as session:
    record = session.run('RETURN "Neo4j connection OK" AS message').single()
    print(record["message"])

driver.close()
print("연결 확인 완료.")


Neo4j connection OK
연결 확인 완료.


**예상 출력:** `Neo4j connection OK` 다음 `연결 확인 완료.`

**Neo4j Browser**에서도 동일한 쿼리로 확인할 수 있습니다(열기 방법은 위 Step 0 참고):

```cypher
RETURN "Neo4j connection OK" AS message;
```

이후 코스에서는 경로를 반환하는 `MATCH … RETURN` 쿼리 실행 시 Browser를 **graph** 뷰(테이블 아님)로 전환하세요.


### Step 0d.1 — `ollama_model_runner.py` 위치 찾기
이 코스는 Module 4/5와 동일한 서브프로세스 runner를 재사용합니다. `Module_8/`, `Module_4/`, 현재 디렉터리 순으로 검색합니다.


In [6]:
# Step 0d.1 — Resolve runner script path
import json
import subprocess
import sys
import tempfile
from pathlib import Path
from typing import Any, List, Optional
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.llms import LLM
def _resolve_ollama_runner_path() -> Path:
    for candidate in (
        MODULE_DIR / "ollama_model_runner.py",
        MODULE_DIR.parent / "Module_4" / "ollama_model_runner.py",
        Path("ollama_model_runner.py"),
    ):
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "ollama_model_runner.py not found. Expected in Module_8/ (see LLM_MODEL_SETUP.md)."
    )
OLLAMA_RUNNER = _resolve_ollama_runner_path()
print("OLLAMA_RUNNER:", OLLAMA_RUNNER)


OLLAMA_RUNNER: /home/ethan/newgen/KMOU_Course/Module_8/ollama_model_runner.py


### Step 0d.2 — `call_ollama_runner()` 헬퍼
1. 프롬프트를 임시 파일에 기록합니다.
2. host, model, temperature, max tokens와 함께 `python ollama_model_runner.py`를 호출합니다.
3. JSON stdout을 파싱해 첫 번째 성공 `response` 문자열을 반환합니다.
노트북마다 HTTP 호출을 중복하지 않고 Ollama 설정을 한 스크립트에 모읍니다.


In [7]:
# Step 0d.2 — Subprocess wrapper around ollama_model_runner.py
def call_ollama_runner(prompt: str, *, model: str | None = None) -> str:
    """Call ollama_model_runner.py (same pattern as Module 4 / Module 5)."""
    model_arg = model or OLLAMA_MODEL
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".txt", delete=False, encoding="utf-8"
    ) as pf:
        path = pf.name
        pf.write(prompt)
    try:
        cmd = [
            sys.executable,
            str(OLLAMA_RUNNER),
            "--host",
            OLLAMA_HOST,
            "--models",
            model_arg,
            "--prompt-file",
            path,
            "--temperature",
            str(OLLAMA_TEMPERATURE),
            "--max-tokens",
            str(OLLAMA_MAX_TOKENS),
        ]
        run = subprocess.run(cmd, capture_output=True, text=True)
        if run.returncode != 0:
            raise RuntimeError(
                f"ollama_model_runner.py exit {run.returncode}\nSTDERR:\n{run.stderr[:4000]}"
            )
        payload = json.loads(run.stdout)
        outputs = payload.get("outputs") or []
        if not outputs:
            raise RuntimeError("Runner JSON contained no outputs")
        first = outputs[0]
        if first.get("status") != "ok":
            raise RuntimeError(f"Ollama runner error: {first}")
        return (first.get("response") or "").strip()
    finally:
        Path(path).unlink(missing_ok=True)


### Step 0d.3 — LangChain `OllamaRunnerLLM` 클래스
`LLMGraphTransformer`는 LangChain **`LLM`**(또는 chat model)을 기대합니다. 이 얇은 래퍼는 `_call()`을 `call_ollama_runner()`로 전달합니다.


In [8]:
# Step 0d.3 — LangChain LLM adapter
class OllamaRunnerLLM(LLM):
    """LangChain LLM that delegates to ollama_model_runner.py."""
    model: str = OLLAMA_MODEL
    temperature: float = OLLAMA_TEMPERATURE
    max_tokens: int = OLLAMA_MAX_TOKENS
    @property
    def _llm_type(self) -> str:
        return "ollama_runner"
    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        return call_ollama_runner(prompt, model=self.model)


### Step 0d.4 — 이후에 사용할 `llm` 객체 생성
추출 전에 Step 0d 셀을 모두 실행하세요. Ollama의 경우 `ollama serve`가 실행 중이고 모델이 pull되어 있어야 합니다(`ollama pull ...`).


In [9]:
# Step 0d.4 — Select backend and instantiate llm
if LLM_BACKEND == "openai":
    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError("OPENAI_API_KEY is required when LLM_BACKEND=openai")
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    print(f"OpenAI 모델 사용: {OPENAI_MODEL}")
elif LLM_BACKEND == "ollama":
    llm = OllamaRunnerLLM()
    print(f"runner를 통해 Ollama 사용: {OLLAMA_MODEL} @ {OLLAMA_HOST}")
    print("Ollama 실행 확인: ollama serve")
else:
    raise ValueError("Set LLM_BACKEND to 'ollama' or 'openai'")


runner를 통해 Ollama 사용: llama3.1:8b @ http://localhost:11434
Ollama 실행 확인: ollama serve


### Step 0e — `ollama_model_runner.py` 스모크 테스트

그래프 추출 전에 한 번 실행하세요. 실패하면 **`LLM_MODEL_SETUP.md`**에서 Ollama 설정을 수정하세요.


In [10]:
# Step 0e — Smoke test (ollama backend only)
if LLM_BACKEND == "ollama":
    _smoke = call_ollama_runner("Reply with exactly: Ollama OK", model=OLLAMA_MODEL)
    print("Ollama runner 스모크 테스트:", _smoke[:200])
    if "ok" not in _smoke.lower():
        print("경고: 예상치 못한 응답 — 모델과 OLLAMA_HOST를 확인하세요")
    print("LLMGraphTransformer용 LLM 준비 완료 (backend=ollama)")
else:
    print("건너뜀 — OpenAI 백엔드 선택됨")


Ollama runner 스모크 테스트: Ollama OK
LLMGraphTransformer용 LLM 준비 완료 (backend=ollama)


---

# 01. 지식 그래프

## 1.1 지식 그래프란

**지식 그래프(knowledge graph)**는 도메인에 대한 **사실(fact)**을 그래프 데이터베이스 형태로 표현한 것입니다:

- **노드(node)** = 엔티티(예: `Port`, `Organization`, `Canal`, `Country`)
- **관계(relationship)** = 유형이 지정된 연결(예: `LOCATED_IN`, `OPERATED_BY`, `CONNECTS`)
- **속성(property)** = 노드 또는 관계의 속성(예: `name`, `country`, `throughput`)

### 비정형 vs 정형

| 비정형 | 정형(Neo4j) |
|--------------|-------------------------|
| PDF, 이메일, 보고서의 문단 | 레이블이 있는 노드와 관계 |
| SQL만으로는 조회가 어려움 | Cypher로 쉽게 순회 |
| 모호한 지칭("그 항구") | 명시적 엔티티(`Port_of_Rotterdam`) |

### 이 코스에서의 지식 그래프 활용

**비정형 텍스트**(DBpedia 스타일 초록)에서 시작해 **LLM**으로 엔티티와 관계를 제안하고, **Neo4j**에 저장한 뒤 Cypher로 결과 그래프를 **조회·탐색**합니다.


## 1.2 이점과 과제

### 이점

| 이점 | 중요한 이유 |
|---------|----------------|
| **연결된 맥락** | 관계가 고립된 텍스트 청크 이상의 의미를 담음 |
| **설명 가능성** | 답변을 그래프 경로로 추적 가능 |
| **재사용 가능한 데이터 모델** | 앱·API용 안정적인 레이블과 관계 유형 |
| **향상된 GenAI(GraphRAG)** | 청크 벡터 검색과 그래프 순회 결합 |

### 과제

| 과제 | 실무적 영향 |
|-----------|------------------|
| **스키마 설계** | 일관되지 않은 레이블은 조회 품질 저하 |
| **추출 품질** | LLM이 엔티티·관계를 환각(hallucinate)할 수 있음 |
| **데이터 정제** | 중복 엔티티·모호한 이름은 검토 필요 |
| **운영 비용** | LLM 추출·임베딩 생성에 시간·연산 필요 |

이 모듈에서는 **스키마 가이드 추출**과 **Cypher 검증 쿼리**로 위 위험을 줄입니다.


## 1.3 지식 그래프 탐색

비정형 텍스트로 그래프를 만들기 전에 Neo4j에서 **작은 참조 그래프**를 탐색합니다.

### 학습 목표

- 기본 Cypher `MATCH` 패턴 실행
- 노드 레이블, 관계 유형, 속성 읽기
- Neo4j Browser에서 부분 그래프 시각화

> Step 0(Neo4j 연결 확인) 후 다음 셀을 실행하세요.


### 1.3a — 참조 노드 시드(재실행 안전)
항구, 조직, 운하, 국가로 구성된 **작은** 해양 테마 그래프를 만듭니다. `CourseKG` 마커 노드는 나중에 선택적 정리를 위해 이 랩 데이터를 태그합니다.


In [11]:
# 1.3a — Seed reference graph (write)
from neo4j import GraphDatabase
seed_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
seed_cypher = [
    '''
    MERGE (demo:CourseKG {module: 'Module_8', lab: 'reference_explore'})
    ''',
    '''
    MERGE (p1:Port {id: 'Port_of_Rotterdam'})
    SET p1.country = 'Netherlands'
    MERGE (p2:Port {id: 'Port_of_Busan'})
    SET p2.country = 'South Korea'
    MERGE (o:Organization {id: 'Maersk'})
    SET o.country = 'Denmark'
    MERGE (c:Canal {id: 'Panama_Canal'})
    SET c.country = 'Panama'
    MERGE (p1)-[:LOCATED_IN]->(country1:Country {id: 'Netherlands'})
    MERGE (p2)-[:LOCATED_IN]->(country2:Country {id: 'South Korea'})
    MERGE (o)-[:OPERATES_IN]->(p1)
    MERGE (o)-[:USES_ROUTE]->(c)
    ''',
]
with seed_driver.session(database=NEO4J_DATABASE) as session:
    for q in seed_cypher:
        session.run(q)
print("참조 그래프 시드 완료.")


참조 그래프 시드 완료.


### 1.3b — 시드된 노드·관계 확인
노트북에서 데이터를 확인한 뒤 **Neo4j Browser**를 열고 경로를 시각화하세요. 예:

```cypher
MATCH p=(:Port)-[*1..2]-() RETURN p LIMIT 25
```


In [12]:
# 1.3b — Query reference graph
with seed_driver.session(database=NEO4J_DATABASE) as session:
    rows = session.run(
        '''
        MATCH (n)
        WHERE n:Port OR n:Organization OR n:Canal OR n:Country
        RETURN labels(n) AS labels, n.id AS id
        ORDER BY id
        LIMIT 20
        '''
    ).data()
    rels = session.run(
        '''
        MATCH (a)-[r]->(b)
        WHERE type(r) IN ['LOCATED_IN', 'OPERATES_IN', 'USES_ROUTE']
        RETURN a.id AS from, type(r) AS rel, b.id AS to
        LIMIT 20
        '''
    ).data()
seed_driver.close()
print("노드:")
for row in rows:
    print(" ", row)
print("관계:")
for row in rels:
    print(" ", row)


노드:
  {'labels': ['Organization'], 'id': 'A.P. Moller - Maersk'}
  {'labels': ['Organization'], 'id': 'Busan Port Authority'}
  {'labels': ['Organization'], 'id': 'Congress'}
  {'labels': ['Country'], 'id': 'Egypt'}
  {'labels': ['Organization'], 'id': 'International Maritime Organization'}
  {'labels': ['Organization'], 'id': 'Maersk'}
  {'labels': ['Organization', 'LlamaIndexLab'], 'id': 'Maersk'}
  {'labels': ['Organization', 'LangChainLab'], 'id': 'Maersk'}
  {'labels': ['Organization'], 'id': 'Maersk Line'}
  {'labels': ['Organization'], 'id': 'Mediterranean Shipping Company'}
  {'labels': ['Country', 'LlamaIndexLab'], 'id': 'Netherlands'}
  {'labels': ['Country', 'LangChainLab'], 'id': 'Netherlands'}
  {'labels': ['Country'], 'id': 'Netherlands'}
  {'labels': ['Organization'], 'id': 'PSA International'}
  {'labels': ['Country'], 'id': 'Panama'}
  {'labels': ['Canal'], 'id': 'Panama Canal'}
  {'labels': ['Organization'], 'id': 'Panama Canal Authority'}
  {'labels': ['Canal', 'Lang

## 1.4 지식 그래프 활용 사례

**관계가 중요할 때** 여러 산업에서 지식 그래프가 사용됩니다:

| 활용 사례 | 예시 질문 |
|----------|------------------|
| **검색·발견** | 유럽 항구를 운영하는 조직은? |
| **추천** | 자주 함께 구매되는 제품은? |
| **리스크·컴플라이언스** | 플래그된 엔티티와 연결된 공급업체는? |
| **운영** | 교란된 운하 경로에 의존하는 자산은? |
| **GenAI / GraphRAG** | 유사 텍스트뿐 아니라 그래프 맥락으로 질문에 답변 |

### GenAI 관련 활용(이 코스)

- **어휘 그래프(lexical graph)** 구축 — 문서/청크 + 임베딩
- **엔티티 그래프(entity graph)** 구축 — 추출된 노드와 관계
- 근거 있는 검색을 위해 청크와 엔티티 연결

Neo4j **LLM Knowledge Graph Builder**가 이 파이프라인을 자동화합니다. Python으로도 동일한 흐름을 구축합니다.


---

# 02. LLM Graph Builder

## 2.1 LLM으로 지식 그래프 구축하는 방법

일반적인 LLM 기반 구축 파이프라인:

1. 비정형 소스 수집(**Ingest**) — 텍스트 파일, PDF, 웹 페이지
2. 처리·임베딩을 위한 텍스트 **청킹(chunking)**
3. LLM으로 엔티티·관계 **추출(extract)**
4. 스키마에 맞게 레이블 **정규화**(선택이지만 권장)
5. Neo4j에 노드·관계 **적재(load)**
6. Cypher와 시각 탐색으로 **검증(validate)**

```mermaid
flowchart LR
  A[Unstructured text] --> B[Chunking]
  B --> C[LLM extraction]
  C --> D[Schema mapping]
  D --> E[(Neo4j graph)]
  E --> F[Cypher exploration]
```

### 이 랩의 데이터셋

| 파일 | 설명 |
|------|-------------|
| **`data/dbpedia_course_corpus.txt`** | **기본값** — 영어 DBpedia 초록 100개(~100 KB) |
| `data/dbpedia_maritime_corpus.txt` | 해양·물류 중심 기사 37개 |
| `data/DATASETS.md` | 출처, 라이선스, 재빌드·다운로드 스크립트 |

출처: [DBpedia-Summarizer en_100_summaries.csv](https://github.com/dice-group/DBpedia-Summarizer/blob/master/data_eval/en_100_summaries.csv) (CC BY-SA).

로컬에서 코퍼스 재빌드:

```bash
python Module_8/scripts/build_dbpedia_corpus.py
```

선택: 더 큰 해양 코퍼스 다운로드(인터넷 필요):

```bash
python Module_8/scripts/download_dbpedia_maritime_sparql.py --limit 150
```

명명된 엔티티, 조직, 위치, 암시적 관계("headquartered in", "operated by", "managed by" 등)를 찾아보세요.


### Step 2a — 랩 코퍼스 로드

Step 0b.3에서 설정한 `SAMPLE_TEXT_PATH`에서 DBpedia 플레인 텍스트 코퍼스를 읽습니다. 미리보기로 인코딩과 대략적 크기를 LLM 추출 전에 확인할 수 있습니다.


In [13]:
# Step 2a — Read sample unstructured text
raw_text = SAMPLE_TEXT_PATH.read_text(encoding="utf-8")
print(f"문자 수: {len(raw_text)}")
print("--- 미리보기 (처음 500자) ---")
print(raw_text[:500])


문자 수: 99962
--- 미리보기 (처음 500자) ---
DBpedia Course Corpus (100 articles) — Module 8 Knowledge Graph Lab

License note: Abstracts derived from DBpedia / Wikipedia content (CC BY-SA).
Source package: dice-group/DBpedia-Summarizer (en_100_summaries.csv) + curated maritime entries.

[1] 90_BC
Year 90 BC was a year of the pre-Julian Roman calendar. At the time it was known as the Year of the Consulship of Caesar and Lupus (or, less frequently, year 664 Ab urbe condita). The denomination 90 BC for this year has been used since the early


### Step 2a-helper — 코퍼스를 기사별로 분할

코퍼스 파일은 `[1] Port_of_Rotterdam` 같은 마커를 사용합니다. Ollama로 **100개 기사 전체를 하나의 `Document`로** 넘기지 마세요 — 모델 출력이 잘리고 **노드 0개**가 자주 발생합니다. 기사당 하나의 `Document`를 만들고 랩에서 처리 개수를 제한합니다.


In [14]:
# Step 2a-helper — Parse numbered articles from corpus text
import re


def parse_corpus_articles(text: str) -> list[dict]:
    """Split Module_8 corpus files on [N] Title lines."""
    pattern = re.compile(r"^\[(\d+)\]\s+(.+)$", re.MULTILINE)
    matches = list(pattern.finditer(text))
    if not matches:
        raise ValueError(
            "No [N] Title markers found. Expected dbpedia_*_corpus.txt format."
        )
    articles = []
    for i, m in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[m.end() : end].strip()
        articles.append(
            {
                "index": int(m.group(1)),
                "title": m.group(2).strip(),
                "text": f"{m.group(0).strip()}\n\n{body}",
            }
        )
    return articles


def articles_to_documents(
    articles: list[dict],
    *,
    source_name: str,
    max_articles: int,
    course: str = "Module_8",
) -> list:
    from langchain_core.documents import Document

    limited = articles[:max_articles]
    return [
        Document(
            page_content=a["text"],
            metadata={
                "source": source_name,
                "course": course,
                "article_index": a["index"],
                "title": a["title"],
            },
        )
        for a in limited
    ]


_all_articles = parse_corpus_articles(raw_text)
print(f"{SAMPLE_TEXT_PATH.name}에서 {len(_all_articles)}개 기사 파싱")
print("처음 제목:", [a["title"] for a in _all_articles[:5]])


dbpedia_course_corpus.txt에서 100개 기사 파싱
처음 제목: ['90_BC', 'Heinrich_Hertz', 'Olof_Palme', 'Seventh_Amendment_to_the_United_States_Constitution', 'Red-eye_effect']


---

## 2.2 Neo4j LLM Graph Builder(웹 애플리케이션)

이 단계는 **수동**(브라우저)입니다. Section 2.2b에서 코드로 작성할 것과 동일한 파이프라인을 익힙니다.

### 3.1 애플리케이션 열기

1. 개요: [Neo4j Labs — LLM Graph Builder](https://neo4j.com/labs/genai-ecosystem/llm-graph-builder/)
2. 호스팅 앱: [https://llm-graph-builder.neo4jlabs.com/](https://llm-graph-builder.neo4jlabs.com/)

### 3.2 Neo4j 연결

**`NEO4J_SETUP.md`**의 URI, 사용자명, 비밀번호를 입력하거나 Aura 자격 증명 파일을 끌어다 놓으세요. LLM 공급자/모델 설정은 **`LLM_MODEL_SETUP.md`**를 따르세요.

### 3.3 비정형 데이터 수집

`data/dbpedia_course_corpus.txt`(또는 `dbpedia_maritime_corpus.txt`)를 업로드하거나 텍스트를 붙여넣습니다. 상태가 **New**가 될 때까지 기다립니다.

### 3.4 LLM 및 스키마 구성(선택)

LLM 공급자를 선택합니다. 선택적으로 허용 노드 레이블과 관계 유형을 정의합니다.

### 3.5 그래프 생성

파일 선택 → **Generate Graph**. 청킹, 임베딩, 추출, 적재를 확인합니다.

### 3.6 결과 탐색

- UI에서 그래프 미리보기
- Neo4j Browser에서: `MATCH (n) RETURN labels(n), count(*)`
- 채팅 시도: *Which organizations are connected to major ports?*

### 3.7 일반적인 그래프 계층

| 계층 | 목적 |
|-------|---------|
| Lexical graph | Document → Chunk(+ 임베딩) |
| Entity graph | 도메인 엔티티와 관계 |
| Links | 언급된 엔티티에 연결된 청크 |

> localhost Neo4j는 호스팅 앱에서 접근하지 못할 수 있습니다 — 필요 시 Section 2.2b만 사용하세요.


## 2.2b 프로그래밍 방식 구축(노트북 경로)
웹 앱과 이 노트북은 동일한 핵심 아이디어를 공유합니다: LangChain **`LLMGraphTransformer`**.
1. 텍스트를 LangChain `Document`로 감싸기
2. `LLMGraphTransformer` 실행 → `GraphDocument` 객체
3. `Neo4jGraph`로 Neo4j에 기록
랩 데이터는 정리를 위해 **`CourseKG`** 마커 노드로 태그됩니다.
### 노트북 단계(순서대로 실행)
| Step | 내용 |
|------|------|
| **2.2b-1** | 코퍼스에서 `Document` |
| **2.2b-2** | 오픈 스키마 transformer 구성 |
| **2.2b-3** | LLM 추출(Ollama에서는 느림) |
| **2.2b-4** | Neo4j에 적재 |
| **2.2b-5** | Cypher 요약 카운트 |


#### 2.2b-1 — 기사당 하나의 `Document`

**Step 2a-helper** 이후 각 기사는 별도 `Document`(보통 300–2,500자)입니다. `LAB_MAX_ARTICLES`(기본값 **5**)가 Section 2.2b의 LLM 호출 수를 제한합니다.


In [15]:
# 2.2b-1 — Build Document list (one article per Document)
documents = articles_to_documents(
    _all_articles,
    source_name=SAMPLE_TEXT_PATH.name,
    max_articles=LAB_MAX_ARTICLES,
)
print("오픈 스키마 추출용 Document:", len(documents))
for d in documents[:3]:
    print(
        " ",
        d.metadata["title"],
        "| 문자:",
        len(d.page_content),
    )


오픈 스키마 추출용 Document: 5
  90_BC | 문자: 364
  Heinrich_Hertz | 문자: 504
  Olof_Palme | 문자: 2663


#### 2.2b-2 — `LLMGraphTransformer` 구성(오픈 스키마)
`allowed_nodes=None`이면 LLM이 레이블을 자유롭게 만들 수 있습니다(탐색에는 좋고 프로덕션에는 노이즈). `ignore_tool_usage=True`는 일부 로컬 모델이 잘 지원하지 않는 tool-calling 경로를 피합니다.


In [16]:
# 2.2b-2 — Open-schema transformer
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=None,
    allowed_relationships=None,
    node_properties=False,
    ignore_tool_usage=True,
)
print("LLMGraphTransformer 준비 완료 (오픈 스키마).")


LLMGraphTransformer 준비 완료 (오픈 스키마).


/tmp/ipykernel_430331/772334155.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


#### 2.2b-3 — LLM 추출 실행(기사별)

> **소요 시간:** 기사마다 Ollama 호출 1회. `LAB_MAX_ARTICLES=5`이면 CPU에서 수 분 예상.

> **팁:** 해양 테마 트리플을 원하면 Step 0b.3에서 `SAMPLE_TEXT_PATH`를 `dbpedia_maritime_corpus.txt`로 설정하세요.

모든 기사에서 **노드 0개**이면 `ollama serve`, 모델 이름, `OLLAMA_MAX_TOKENS`(4096 시도)를 확인하세요.


In [17]:
# 2.2b-3 — Extract entities and relationships (per article)
graph_documents = []
for i, doc in enumerate(documents, start=1):
    title = doc.metadata.get("title", f"article_{i}")
    print(f"\n[{i}/{len(documents)}] 추출 중: {title} ({len(doc.page_content)}자)")
    batch = llm_transformer.convert_to_graph_documents([doc])
    graph_documents.extend(batch)
    gd_i = batch[0]
    print(f"  -> 노드: {len(gd_i.nodes)}, 관계: {len(gd_i.relationships)}")

total_nodes = sum(len(gd.nodes) for gd in graph_documents)
total_rels = sum(len(gd.relationships) for gd in graph_documents)
print(f"\n{len(graph_documents)}개 graph document 합계: 노드 {total_nodes}개, 관계 {total_rels}개")

if total_nodes == 0:
    print(
        "경고: 추출된 노드 없음. Ollama 실행 여부 확인, OLLAMA_MAX_TOKENS 증가, "
        "또는 프롬프트 크기 축소. SAMPLE_TEXT_PATH = dbpedia_maritime_corpus.txt 시도"
    )
else:
    gd = graph_documents[0]
    print("\n첫 비어 있지 않은 배치 샘플 (노드/관계 각 8개):")
    for n in gd.nodes[:8]:
        print("  노드:", n.id, n.type, n.properties)
    for r in gd.relationships[:8]:
        print("  관계:", r.source.id, "->", r.type, "->", r.target.id)



[1/5] 추출 중: 90_BC (364자)
  -> 노드: 3, 관계: 2

[2/5] 추출 중: Heinrich_Hertz (504자)
  -> 노드: 5, 관계: 4

[3/5] 추출 중: Olof_Palme (2663자)
  -> 노드: 5, 관계: 4

[4/5] 추출 중: Seventh_Amendment_to_the_United_States_Constitution (1958자)
  -> 노드: 8, 관계: 7

[5/5] 추출 중: Red-eye_effect (657자)
  -> 노드: 4, 관계: 4

5개 graph document 합계: 노드 25개, 관계 21개

첫 비어 있지 않은 배치 샘플 (노드/관계 각 8개):
  노드: Year 90 BC Time Period {}
  노드: 664 Ab urbe condita Date System {}
  노드: The Year of the Consulship of Caesar and Lupus Event {}
  관계: Year 90 BC -> KNOWN_AS -> The Year of the Consulship of Caesar and Lupus
  관계: Year 90 BC -> EQUIVALENT_TO -> 664 Ab urbe condita


#### 2.2b-4 — Neo4j 연결 및 그래프 영속화
`Neo4jGraph.add_graph_documents()`는 노드, 관계를 만들고(`include_source=True`일 때) 소스 문서로 다시 연결합니다.


In [18]:
# 2.2b-4 — Write GraphDocuments to Neo4j
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)
graph.query(
    '''
    MERGE (m:CourseKG {module: 'Module_8', lab: 'building_kg_with_llms'})
    SET m.updatedAt = datetime()
    RETURN m.module AS module
    '''
)
graph.add_graph_documents(graph_documents, include_source=True)
print("Graph document가 Neo4j에 기록되었습니다.")


Graph document가 Neo4j에 기록되었습니다.


#### 2.2b-5 — 적재 구조 요약
노드 레이블·관계 유형별 빠른 카운트 — 웹 UI 적재 후 Neo4j Browser에서 하는 것과 동일한 확인입니다.


In [19]:
# 2.2b-5 — Post-load summary queries
summary = graph.query(
    '''
    MATCH (n)
    WHERE NOT n:CourseKG
    RETURN labels(n)[0] AS label, count(*) AS count
    ORDER BY count DESC
    LIMIT 15
    '''
)
print("레이블별 노드 수:")
for row in summary:
    print(f"  {row['label']}: {row['count']}")
rels = graph.query(
    '''
    MATCH ()-[r]->()
    RETURN type(r) AS relType, count(*) AS count
    ORDER BY count DESC
    LIMIT 15
    '''
)
print("\n관계 유형별 수:")
for row in rels:
    print(f"  {row['relType']}: {row['count']}")


레이블별 노드 수:
  WarehouseInventoryLab: 2215
  Document: 17
  Organization: 14
  Country: 10
  Port: 9
  Location: 6
  Person: 6
  LangChainLab: 5
  Canal: 5
  Concept: 5
  Chunk: 4
  RiskCategory: 3
  Group: 2
  Device: 1
  Action: 1

관계 유형별 수:
  IN_WEEK: 1995
  HAS_RISK: 1995
  HAS_SNAPSHOT: 1995
  HAS_INVENTORY_STATUS: 1995
  HAS_FULFILLMENT_STATUS: 1995
  HAS_EQUIPMENT_STATUS: 1995
  HAS_SUPPLIER_TIER: 1995
  MENTIONS: 58
  LOCATED_IN: 13
  OPERATES_IN: 10
  USES_ROUTE: 9
  OPERATED_BY: 8
  HAS_CHUNK: 4
  CONNECTS: 3
  AFFECTS: 2


---

## 2.3 스키마 맞춤 설정

프로덕션 그래프는 레이블과 관계 유형의 **통제된 어휘(controlled vocabulary)**를 사용합니다.

| 노드 레이블 | 예시 |
|-------------|----------|
| `Port`, `Organization`, `Canal`, `Country`, `Location` | Port of Rotterdam, Maersk, Panama Canal |

| 관계 유형 | 패턴 |
|--------------------|---------|
| `LOCATED_IN`, `OPERATED_BY`, `OPERATES_IN`, `USES_ROUTE`, `MANAGED_BY` | 도메인별 |

Neo4j LLM Graph Builder UI의 **엔티티 그래프 추출 설정**과 동일한 개념입니다.

### 노트북 단계

| Step | 내용 |
|------|------|
| **2.3a** | 허용 레이블·관계 유형 정의 |
| **2.3b** | 스키마 가이드 추출 |
| **2.3c** | `lab: schema_guided`로 Neo4j에 적재 |


#### 2.3a — 통제 스키마 정의
**`allowed_nodes`**와 **`allowed_relationships`**를 제한하면 환각 레이블이 줄고 Neo4j LLM Graph Builder 스키마 UI와 맞춰집니다.


In [20]:
# 2.3a — Schema vocabulary
SCHEMA_NODES = ["Port", "Organization", "Canal", "Country", "Location"]
SCHEMA_RELS = [
    "LOCATED_IN",
    "OPERATED_BY",
    "OPERATES_IN",
    "MANAGED_BY",
    "USES_ROUTE",
    "CONNECTS",
]
print("허용 노드:", SCHEMA_NODES)
print("허용 관계:", SCHEMA_RELS)


허용 노드: ['Port', 'Organization', 'Canal', 'Country', 'Location']
허용 관계: ['LOCATED_IN', 'OPERATED_BY', 'OPERATES_IN', 'MANAGED_BY', 'USES_ROUTE', 'CONNECTS']


#### 2.3b — 스키마 가이드 추출

가능하면 **`dbpedia_maritime_corpus.txt`**를 사용합니다(항구, Maersk, 운하 등). Ollama에서는 레이블이 약간 어긋나도 유지되도록 `strict_mode=False`; OpenAI는 `strict_mode=True`와 `node_properties` 사용 가능.

Section 2.2b 오픈 스키마 실행 결과와 합계를 비교하세요.


In [21]:
# 2.3b — Run schema-guided LLMGraphTransformer
# Prefer maritime articles for Port / Organization / Canal schema
_maritime_path = MODULE_DIR / "data" / "dbpedia_maritime_corpus.txt"
if _maritime_path.is_file():
    _schema_articles = parse_corpus_articles(_maritime_path.read_text(encoding="utf-8"))
    _schema_source = _maritime_path.name
else:
    _schema_articles = _all_articles
    _schema_source = SAMPLE_TEXT_PATH.name

schema_documents = articles_to_documents(
    _schema_articles,
    source_name=_schema_source,
    max_articles=LAB_SCHEMA_ARTICLES,
    course="Module_8_schema",
)
print(f"스키마 추출: {_schema_source}에서 {len(schema_documents)}개 기사")

_schema_kwargs = dict(
    llm=llm,
    allowed_nodes=SCHEMA_NODES,
    allowed_relationships=SCHEMA_RELS,
    ignore_tool_usage=True,
    # Ollama JSON mode: strict filtering often drops all triples if labels differ slightly
    strict_mode=(LLM_BACKEND == "openai"),
    additional_instructions=(
        "Focus on ports, shipping companies, canals, countries, and logistics locations. "
        "Use only the allowed node labels and relationship types from the schema."
    ),
)
if LLM_BACKEND == "openai":
    _schema_kwargs["node_properties"] = ["country", "date"]
else:
    print(
        "참고: node_properties 생략 — Ollama runner는 function calling 미지원. "
        "strict_mode=False로 유효한 해양 트리플이 모두 걸러지지 않도록 함."
    )

schema_transformer = LLMGraphTransformer(**_schema_kwargs)
print("스키마 가이드 추출 실행 중...")
schema_graph_documents = []
for i, doc in enumerate(schema_documents, start=1):
    title = doc.metadata.get("title", f"article_{i}")
    print(f"  [{i}/{len(schema_documents)}] {title}")
    batch = schema_transformer.convert_to_graph_documents([doc])
    schema_graph_documents.extend(batch)
    b = batch[0]
    print(f"      노드={len(b.nodes)}, 관계={len(b.relationships)}")

sgd_nodes = sum(len(gd.nodes) for gd in schema_graph_documents)
sgd_rels = sum(len(gd.relationships) for gd in schema_graph_documents)
print(f"스키마 가이드 합계: 노드 {sgd_nodes}개, 관계 {sgd_rels}개")

if sgd_nodes == 0:
    print("\n디버그: 첫 스키마 document의 원시 LLM 출력 (처음 1500자):")
    _raw = schema_transformer.chain.invoke({"input": schema_documents[0].page_content})
    if not isinstance(_raw, str):
        _raw = getattr(_raw, "content", str(_raw))
    print(_raw[:1500])


스키마 추출: dbpedia_maritime_corpus.txt에서 8개 기사
참고: node_properties 생략 — Ollama runner는 function calling 미지원. strict_mode=False로 유효한 해양 트리플이 모두 걸러지지 않도록 함.
스키마 가이드 추출 실행 중...
  [1/8] Port_of_Singapore
      노드=3, 관계=2
  [2/8] Port_of_Rotterdam
      노드=3, 관계=2
  [3/8] Port_of_Busan
      노드=3, 관계=2
  [4/8] Maersk
      노드=4, 관계=2
  [5/8] Mediterranean_Shipping_Company
      노드=3, 관계=2
  [6/8] Panama_Canal
      노드=3, 관계=2
  [7/8] Suez_Canal
      노드=5, 관계=4
  [8/8] International_Maritime_Organization
      노드=3, 관계=2
스키마 가이드 합계: 노드 27개, 관계 18개


#### 2.3c — 스키마 가이드 그래프 적재
메타데이터 `lab: schema_guided`는 오픈 스키마 랩 적재와 구분해 정리 쿼리에 사용합니다.


In [22]:
# 2.3c — Persist schema-guided GraphDocuments
graph.query(
    '''
    MERGE (m:CourseKG {module: 'Module_8', lab: 'schema_guided'})
    SET m.updatedAt = datetime()
    '''
)
for doc in schema_graph_documents:
    doc.source.metadata["lab"] = "schema_guided"
graph.add_graph_documents(schema_graph_documents, include_source=True)
print("스키마 가이드 그래프 로드 완료.")


스키마 가이드 그래프 로드 완료.


## 2.4 그래프 완성

다음이 모두 참이면 그래프가 **완성**된 것입니다:

1. **엔티티 적재** — Neo4j에 예상 레이블의 노드가 보임
2. **관계 적재** — 유형이 지정된 엣지가 올바른 엔티티를 연결
3. **출처 보존** — 소스 문서/청크 링크 존재(`include_source=True` 사용 시)
4. **스키마 패스 완료** — 스키마 가이드 추출이 치명적 오류 없이 끝남
5. **건전성 검사 통과** — 레이블·관계 카운트가 합리적

### 완료 체크리스트

- [ ] 레이블 카운트 쿼리 실행 (`MATCH (n) RETURN labels(n), count(*)`)
- [ ] 관계 카운트 쿼리 실행 (`MATCH ()-[r]->() RETURN type(r), count(*)`)
- [ ] Neo4j Browser graph 뷰에서 엔티티 3개 스팟 체크
- [ ] Section 03에서 Cypher로 도메인 질문 1개 실행

카운트가 0이거나 레이블이 일관되지 않으면 **2.3 스키마 맞춤 설정**을 다시 보고 추출을 재실행하세요.


---

# 03. 지식 그래프 탐색

## 3.1 Cypher로 조회

LLM을 호출하지 않고 Cypher로 질문에 답합니다.

### 시작 패턴

- `MATCH (n:Port) RETURN n LIMIT 10`
- `MATCH (a)-[r]->(b) RETURN a.id, type(r), b.id LIMIT 20`
- `MATCH p=(a)-[*1..3]-(b) RETURN p LIMIT 10`


#### 3.1a — 패턴: 조직이 항구에서 운영
명시적 트리플 `(Organization)-[:OPERATES_IN]->(Port)` 반환.


In [23]:
# 3.1a — Organization → Port
rows = graph.query(
    '''
    MATCH (o:Organization)-[r:OPERATES_IN]->(p:Port)
    RETURN o.id AS organization, p.id AS port
    LIMIT 20
    '''
)
if rows:
    for row in rows:
        print(row)
else:
    print("OPERATES_IN 행이 아직 없습니다 — 오픈 스키마 추출 또는 참조 시드 그래프를 시도하세요.")


{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}


#### 3.1b — 패턴: 항구 위치 및 선택적 운영자
항구에 연결된 조직이 없을 수 있을 때 **`OPTIONAL MATCH`** 사용.


In [24]:
# 3.1b — Port, country, optional organization
for row in graph.query(
    '''
    MATCH (p:Port)-[:LOCATED_IN]->(c:Country)
    OPTIONAL MATCH (o:Organization)-[:OPERATES_IN]->(p)
    RETURN p.id AS port, c.id AS country, o.id AS organization
    LIMIT 15
    '''
):
    print(row)


{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Busan', 'country': 'South Korea', 'organization': None}
{'port': 'Port of Rotterdam', 'country': 'Netherlands', 'organization': None}
{'port': 'Port of Rotterdam', 'country': 'Netherlands', 'organization': None}
{'port': 'Port of Rotterdam', 'country': 'Netherlands', 'organization': None}
{'port': 'Port of Busan', 'country': 'South Korea', 'organization': None}
{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Singapore', 'country': 'Singapore', 'organization': None}
{'port': 'Port_of_Rotterdam', 'country': 'Netherla

#### 3.1c — 이름 조각으로 검색
매개변수 `CONTAINS`는 노트북에서 대화형 탐색에 유용합니다.


In [25]:
# 3.1c — Find entities containing a term
SEARCH_TERM = "Rotterdam"
for row in graph.query(
    '''
    MATCH (n)
    WHERE n.id CONTAINS $term
    RETURN labels(n) AS labels, n.id AS id
    LIMIT 15
    ''',
    params={"term": SEARCH_TERM},
):
    print(row)


{'labels': ['Port'], 'id': 'Port_of_Rotterdam'}
{'labels': ['Port'], 'id': 'Port of Rotterdam'}
{'labels': ['Organization'], 'id': 'Port of Rotterdam Authority'}
{'labels': ['Port', 'LlamaIndexLab'], 'id': 'Port_of_Rotterdam'}
{'labels': ['Port', 'LangChainLab'], 'id': 'Port_of_Rotterdam'}


#### 3.1d — 두 엔티티 간 최단 경로
가변 길이 경로 `-[*..6]-` 사용. 경로가 없으면 Neo4j는 오류가 아니라 빈 결과를 반환합니다.


In [26]:
# 3.1d — Shortest path example
start_name = "Port_of_Rotterdam"
end_name = "Panama_Canal"
result = graph.query(
    '''
    MATCH (a {id: $start}), (b {id: $end})
    MATCH p = shortestPath((a)-[*..6]-(b))
    RETURN [n IN nodes(p) | coalesce(n.id, toString(id(n)))] AS path,
           [r IN relationships(p) | type(r)] AS relTypes
    ''',
    params={"start": start_name, "end": end_name},
)
if result:
    print("경로:", result[0]["path"])
    print("관계 유형:", result[0]["relTypes"])
else:
    print(f"{start_name}와 {end_name} 사이에 경로 없음")


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=53, offset=139>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 139, 'line': 4, 'column': 53}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (a {id: $start}), (b {id: $end})\n    MATCH p = shortestPath((a)-[*..6]-(b))\n    RETURN [n IN nodes(p) | coalesce(n.id, toString(id(n)))] AS path,\n           [r IN relationships(p) | type(r)] AS relTypes\n    '


경로: ['Port_of_Rotterdam', 'Maersk', 'Panama_Canal']
관계 유형: ['OPERATES_IN', 'USES_ROUTE']


## 3.2 지식 그래프 탐색

탐색은 구조를 검증하고 시각·프로그램 방식으로 인사이트를 발견하는 것입니다.

### Neo4j Browser / Bloom

1. Neo4j 인스턴스 열기(Aura, Desktop, Docker Browser URL)
2. 그래프 스타일 쿼리 실행, 예:

```cypher
MATCH p=(:Port)-[*1..2]-()
RETURN p
LIMIT 25
```

3. 노드를 클릭해 속성 확인(`id`, `country` 등)

### 확인할 것

- 항구·조직 엔티티가 예상대로 연결되어 있는가?
- 관계 유형이 스키마와 일치하는가?
- 고아 노드(관계 없음)가 있는가?


#### 3.2a — 고아 노드(관계 없음)
고아는 추출 노이즈나 스키마에 빠진 관계 유형을 나타내는 경우가 많습니다.


In [27]:
# 3.2a — Orphan node sample
orphan_nodes = graph.query(
    '''
    MATCH (n)
    WHERE NOT (n)--() AND NOT n:CourseKG
    RETURN labels(n) AS labels, coalesce(n.id, toString(id(n))) AS id
    LIMIT 20
    '''
)
print("고아 노드 (샘플):")
if orphan_nodes:
    for row in orphan_nodes:
        print(" ", row)
else:
    print("  샘플에서 없음 (좋은 신호).")


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=57, offset=112>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 112, 'line': 4, 'column': 57}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (n)\n    WHERE NOT (n)--() AND NOT n:CourseKG\n    RETURN labels(n) AS labels, coalesce(n.id, toString(id(n))) AS id\n    LIMIT 20\n    '


고아 노드 (샘플):
  {'labels': ['Document'], 'id': 'bc8a4d5feaf6c9d0eb16543ecbc42866'}
  {'labels': ['Chunk'], 'id': 'f02c41ffe31773d071bc850d6582ba2e'}
  {'labels': ['Chunk'], 'id': 'c3e2bd57957511fc6b22b15be202adbb'}
  {'labels': ['LlamaIndexLab'], 'id': '74'}
  {'labels': ['Chunk'], 'id': 'f21032245792e8fafc7e94cbed7864f8'}
  {'labels': ['Chunk'], 'id': 'b2ed1b839b4fc452dc7170c595dc6864'}
  {'labels': ['LangChainLab'], 'id': '1214'}


#### 3.2b — 고연결(hub) 노드
많은 엣지가 연결된 노드는 주요 항구, 글로벌 조직 등 중요 엔티티인 경우가 많습니다.


In [28]:
# 3.2b — Hub nodes by degree
hubs = graph.query(
    '''
    MATCH (n)-[r]-()
    WHERE NOT n:CourseKG
    RETURN labels(n)[0] AS label, coalesce(n.id, toString(id(n))) AS id, count(r) AS degree
    ORDER BY degree DESC
    LIMIT 10
    '''
)
print("연결 수가 가장 많은 노드:")
for row in hubs:
    print(" ", row)


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=59, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 4, 'column': 59}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (n)-[r]-()\n    WHERE NOT n:CourseKG\n    RETURN labels(n)[0] AS label, coalesce(n.id, toString(id(n))) AS id, count(r) AS degree\n    ORDER BY degree DESC\n    LIMIT 10\n    '


연결 수가 가장 많은 노드:
  {'label': 'WarehouseInventoryLab', 'id': 'Critical', 'degree': 1726}
  {'label': 'RiskCategory', 'id': 'High Risk', 'degree': 1679}
  {'label': 'WarehouseInventoryLab', 'id': 'Low', 'degree': 991}
  {'label': 'WarehouseInventoryLab', 'id': 'Medium', 'degree': 754}
  {'label': 'WarehouseInventoryLab', 'id': 'Behind', 'degree': 694}
  {'label': 'WarehouseInventoryLab', 'id': 'OnTrack', 'degree': 683}
  {'label': 'WarehouseInventoryLab', 'id': 'Balanced', 'degree': 680}
  {'label': 'WarehouseInventoryLab', 'id': 'High', 'degree': 638}
  {'label': 'WarehouseInventoryLab', 'id': 'AtRisk', 'degree': 618}
  {'label': 'WarehouseInventoryLab', 'id': 'Constrained', 'degree': 556}


### GraphRAG 상기(1.4 활용 사례)

| 기능 | 이점 |
|------------|---------|
| 명시적 관계 | Cypher로 다중 홉 추론 |
| 스키마 | 에이전트·API용 안정 레이블 |
| 출처(provenance) | 답변을 소스 청크에 연결 |
| GraphRAG | LLM 프롬프트에 청크 + 그래프 맥락 |

선택 다음 단계: **`neo4j-graphrag`** (`SimpleKGPipeline`) — `NEO4J_SETUP.md`와 `LLM_MODEL_SETUP.md` 참고.


## 3.3 나만의 지식 그래프 만들기

캡스톤: 자신의 도메인용 그래프를 구축하고 검증합니다.

### 프로젝트 개요

1. 짧은 비정형 소스 선택(500–2000단어)
2. 스키마 정의(노드 레이블 4개 이상, 관계 유형 4개 이상)
3. 그래프 구축(**2.2** UI 또는 **2.2b** + **2.3** 코드)
4. **2.4** 체크리스트로 그래프 완성
5. **3.1**, **3.2**에서 탐색·조회
6. 실제 비즈니스 질문에 답하는 Cypher 3개 작성

### 제안 도메인

- 해양 물류·항구
- 공급망·제조
- 의료 제공자·치료
- 사이버 보안 사고·자산

### 제출물

- 그래프 시각화 스크린샷 또는 export
- Cypher 3개 + 결과 해석
- 스키마 선택·추출 품질에 대한 단락 1개

---

### 선택: 정리


In [29]:
CLEANUP = False

if CLEANUP:
    graph.query("MATCH (m:CourseKG {module: 'Module_8'}) DETACH DELETE m")
    print("정리 실행 완료.")
else:
    print("정리 건너뜀. CourseKG 마커 제거는 CLEANUP=True로 설정하세요.")


정리 건너뜀. CourseKG 마커 제거는 CLEANUP=True로 설정하세요.


---

## 요약

전체 학습 경로를 완료했습니다:

- **01. 지식 그래프** — 개념, 이점/과제, 탐색, 활용 사례
- **02. LLM Graph Builder** — LLM 구축 흐름, Neo4j UI, 스키마 맞춤, 그래프 완성
- **03. 지식 그래프 탐색** — Cypher 조회, 탐색, 캡스톤 프로젝트

**추가 읽을거리**

- [Creating Knowledge Graphs from Unstructured Data](https://neo4j.com/developer/genai-ecosystem/importing-graph-from-unstructured-data/)
- [Introduction to the Neo4j LLM Knowledge Graph Builder](https://neo4j.com/blog/developer/llm-knowledge-graph-builder/)

**설정:** `NEO4J_SETUP.md` 및 `LLM_MODEL_SETUP.md`
